In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.svm import SVR, SVC
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.linear_model import LassoCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [4]:
#load the features
features_df = pd.read_csv("/Users/folasewaabdulsalam/Downloads/EEG_Based_Mental_Workload_Classifier/EEG-Based_Mental_Workload_Classifier/features.csv")
print(f"The Features shape:{features_df.shape}")
features_df.head(5)

The Features shape:(93, 58)


,delta_AF3,delta_F7,delta_F3,delta_FC5,delta_T7,delta_P7,delta_O1,delta_O2,delta_P8,delta_T8,delta_FC6,delta_F4,delta_F8,delta_AF4,theta_AF3,theta_F7,theta_F3,theta_FC5,theta_T7,theta_P7,theta_O1,theta_O2,theta_P8,theta_T8,theta_FC6,theta_F4,theta_F8,theta_AF4,alpha_AF3,alpha_F7,alpha_F3,alpha_FC5,alpha_T7,alpha_P7,alpha_O1,alpha_O2,alpha_P8,alpha_T8,alpha_FC6,alpha_F4,alpha_F8,alpha_AF4,beta_AF3,beta_F7,beta_F3,beta_FC5,beta_T7,beta_P7,beta_O1,beta_O2,beta_P8,beta_T8,beta_FC6,beta_F4,beta_F8,beta_AF4,subject,workload
0,10.725343,5.077131,2.238050,0.697380,56.521224,15.568467,3.971748,1.247739,5.578479,1.934691,0.712844,0.407191,12.795674,4.511928,1.318341,0.705692,8.263090,2.630397,0.928141,0.395330,7.347602,2.303424,0.914592,0.421859,11.204782,4.272083,1.452716,0.560322,5.076533,3.406351,1.213732,0.483922,6.406052,3.570748,1.168518,0.448386,6.494286,2.256176,0.835241,0.380699,27.372321,4.310070,1.152035,0.668551,84.755287,12.084063,2.014611,0.918854,30.620624,9.059456,2.534956,1.105544,12.393469,5.364822,1.685556,0.879012,sub01,hi
1,5516.791766,963.839271,128.723981,12.964473,1396.342661,220.522152,44.662870,6.670238,5436.258042,945.965484,128.996587,13.181641,5289.765038,923.486669,125.214293,12.949809,834160.507942,145756.197419,19838.664266,2054.810706,5234.574359,934.245427,121.218325,13.420974,5408.449855,946.826158,124.216188,13.711892,5448.023987,967.155229,128.424132,13.618631,5232.648600,922.547568,125.430771,13.195478,5272.935324,918.758643,126.813771,13.279635,5662.780441,944.407126,129.740761,14.030177,5531.141189,943.499801,128.402542,13.638912,5296.014472,923.484204,129.758557,13.410973,5471.254522,949.507594,131.673970,14.620083,sub01,lo
2,3.989949,2.608310,1.053837,0.753365,19.472834,10.321517,4.888012,3.832464,4.447023,2.039295,0.922276,0.531007,8.214316,4.451557,2.272394,2.138273,49.848417,32.710839,13.682236,5.822852,2.791738,1.800521,1.206448,0.753437,5.713703,3.039234,1.465519,0.656980,3.709916,2.545335,1.529141,0.662889,4.322220,2.790362,1.657360,0.911896,5.622657,3.061691,1.939932,1.772465,14.007335,5.539859,2.241305,1.426794,5.223372,1.915019,0.895182,0.496758,18.877661,9.902051,3.861640,2.306138,3.648893,2.466794,0.987598,0.551459,sub02,hi
3,2.253481,0.975757,0.694703,0.259047,2.421729,0.950285,0.583619,0.401876,8.895102,1.115093,0.752324,0.231583,1.226110,0.586513,0.516919,0.259382,4.029523,1.406612,1.266557,0.405244,2.414425,0.753805,0.753813,0.185837,2.787471,1.024356,1.359596,0.257515,2.707268,1.199285,1.913981,0.287144,1.371532,1.250067,1.287240,0.246985,1.533410,1.183229,1.041034,0.320760,3.105330,0.966786,0.579054,0.188302,1.135482,0.401280,0.312649,0.120779,4.412904,1.214817,0.641196,0.527638,2.726505,0.995539,0.698546,0.281283,sub02,lo
4,55.389697,9.962111,4.743549,0.646834,103.442932,12.132124,5.344133,0.949898,8.043204,3.059159,3.912413,0.555607,29.964990,4.741650,3.742729,0.687123,17.945655,4.723625,3.920086,0.826001,13.600158,4.121607,9.683179,1.227298,11.132866,4.088251,12.261857,1.124941,10.038657,4.335177,10.751245,1.005637,8.900717,3.063756,6.377781,0.841044,17.790420,4.105695,3.979775,0.725162,23.473292,4.208927,3.335243,0.653718,11.565717,3.191505,3.662566,0.569632,87.510300,12.516137,6.051994,1.143680,26.308322,3.320430,3.043306,0.462703,sub03,hi


In [5]:
#load and merge ratings
ratings = pd.read_csv("/Users/folasewaabdulsalam/Downloads/EEG_Based_Mental_Workload_Classifier/EEG-Based_Mental_Workload_Classifier/ratings.txt", header=None, names=['subject', 'rating_low', 'rating_high'])
#create subject mapping to align with our features dataframe 1 → 'sub01'

subject_mapping = {i: f'sub{i:02d}' for i in range(1,49)}
ratings['subject'] = ratings['subject'].replace(subject_mapping)
print(ratings.head())

  subject  rating_low  rating_high
0   sub01           2            8
1   sub02           1            5
2   sub03           1            5
3   sub04           2            5
4   sub06           4            7


In [8]:
#merging the two datasets

#prepare the ratings in long format
ratings_long = []

for _, row in ratings.iterrows():
    ratings_long.append(
        {'subject': row['subject'], 'workload': 'lo', 'rating': row['rating_low']} #low workload ratings

    )
    ratings_long.append(
        {'subject': row['subject'], 'workload': 'hi', 'rating':row['rating_high']} #high workload ratings
    )
ratings_df = pd.DataFrame(ratings_long)
ratings_df.head()

#merging the long format of the ratings text with the features


data = features_df.merge(ratings_df, on=['subject', 'workload'], how='inner')

print(f"\nMerged data shape: {data.shape}")
print(data['rating'].value_counts().sort_index())
data.head()



Merged data shape: (87, 59)
rating
1    23
2    11
3     7
4     5
5    11
6     6
7    11
8    10
9     3
Name: count, dtype: int64


,delta_AF3,delta_F7,delta_F3,delta_FC5,delta_T7,delta_P7,delta_O1,delta_O2,delta_P8,delta_T8,delta_FC6,delta_F4,delta_F8,delta_AF4,theta_AF3,theta_F7,theta_F3,theta_FC5,theta_T7,theta_P7,theta_O1,theta_O2,theta_P8,theta_T8,theta_FC6,theta_F4,theta_F8,theta_AF4,alpha_AF3,alpha_F7,alpha_F3,alpha_FC5,alpha_T7,alpha_P7,alpha_O1,alpha_O2,alpha_P8,alpha_T8,alpha_FC6,alpha_F4,alpha_F8,alpha_AF4,beta_AF3,beta_F7,beta_F3,beta_FC5,beta_T7,beta_P7,beta_O1,beta_O2,beta_P8,beta_T8,beta_FC6,beta_F4,beta_F8,beta_AF4,subject,workload,rating
0,10.725343,5.077131,2.238050,0.697380,56.521224,15.568467,3.971748,1.247739,5.578479,1.934691,0.712844,0.407191,12.795674,4.511928,1.318341,0.705692,8.263090,2.630397,0.928141,0.395330,7.347602,2.303424,0.914592,0.421859,11.204782,4.272083,1.452716,0.560322,5.076533,3.406351,1.213732,0.483922,6.406052,3.570748,1.168518,0.448386,6.494286,2.256176,0.835241,0.380699,27.372321,4.310070,1.152035,0.668551,84.755287,12.084063,2.014611,0.918854,30.620624,9.059456,2.534956,1.105544,12.393469,5.364822,1.685556,0.879012,sub01,hi,8
1,5516.791766,963.839271,128.723981,12.964473,1396.342661,220.522152,44.662870,6.670238,5436.258042,945.965484,128.996587,13.181641,5289.765038,923.486669,125.214293,12.949809,834160.507942,145756.197419,19838.664266,2054.810706,5234.574359,934.245427,121.218325,13.420974,5408.449855,946.826158,124.216188,13.711892,5448.023987,967.155229,128.424132,13.618631,5232.648600,922.547568,125.430771,13.195478,5272.935324,918.758643,126.813771,13.279635,5662.780441,944.407126,129.740761,14.030177,5531.141189,943.499801,128.402542,13.638912,5296.014472,923.484204,129.758557,13.410973,5471.254522,949.507594,131.673970,14.620083,sub01,lo,2
2,3.989949,2.608310,1.053837,0.753365,19.472834,10.321517,4.888012,3.832464,4.447023,2.039295,0.922276,0.531007,8.214316,4.451557,2.272394,2.138273,49.848417,32.710839,13.682236,5.822852,2.791738,1.800521,1.206448,0.753437,5.713703,3.039234,1.465519,0.656980,3.709916,2.545335,1.529141,0.662889,4.322220,2.790362,1.657360,0.911896,5.622657,3.061691,1.939932,1.772465,14.007335,5.539859,2.241305,1.426794,5.223372,1.915019,0.895182,0.496758,18.877661,9.902051,3.861640,2.306138,3.648893,2.466794,0.987598,0.551459,sub02,hi,5
3,2.253481,0.975757,0.694703,0.259047,2.421729,0.950285,0.583619,0.401876,8.895102,1.115093,0.752324,0.231583,1.226110,0.586513,0.516919,0.259382,4.029523,1.406612,1.266557,0.405244,2.414425,0.753805,0.753813,0.185837,2.787471,1.024356,1.359596,0.257515,2.707268,1.199285,1.913981,0.287144,1.371532,1.250067,1.287240,0.246985,1.533410,1.183229,1.041034,0.320760,3.105330,0.966786,0.579054,0.188302,1.135482,0.401280,0.312649,0.120779,4.412904,1.214817,0.641196,0.527638,2.726505,0.995539,0.698546,0.281283,sub02,lo,1
4,55.389697,9.962111,4.743549,0.646834,103.442932,12.132124,5.344133,0.949898,8.043204,3.059159,3.912413,0.555607,29.964990,4.741650,3.742729,0.687123,17.945655,4.723625,3.920086,0.826001,13.600158,4.121607,9.683179,1.227298,11.132866,4.088251,12.261857,1.124941,10.038657,4.335177,10.751245,1.005637,8.900717,3.063756,6.377781,0.841044,17.790420,4.105695,3.979775,0.725162,23.473292,4.208927,3.335243,0.653718,11.565717,3.191505,3.662566,0.569632,87.510300,12.516137,6.051994,1.143680,26.308322,3.320430,3.043306,0.462703,sub03,hi,5
